## **Analysis of stromal gap distances based on Morphometrics results, WL vs FRL**

Based on output from Morphometrics.
Run on individual, separated membranes, with only stromal gap distances calculated. 

Run on default MM directory tree, i.e.

    ST1/ 
    |__ tomo1 
         |__ *rh9.csv 
    |__ tomo1 
         |__ *rh9.csv etc

#### Data import:

In [ ]:
from pathlib import Path
import os
import re
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 100)

CONDITION_DIRS = {
    "WL": Path("path/to/Morphometrics/"),
    "FRL": Path("path/to/Morphometrics"),
}

CONDITION_COLORS = {
    "WL": "#3986a1",
    "FRL": "#a41858",
}

# use plain AVV files, not excluding1borders
USE_EXCLUDING_BORDERS = False

# subtract this once from every measured distance, unit: nm
MEMBRANE_THICKNESS_NM = 4.95

# Maximum distance cutoff for filtering; set to None to keep all
DISTANCE_CUTOFF_NM = 250

OUTDIR = Path("./morphometrics_distance_analysis")
OUTDIR.mkdir(exist_ok=True)

#### Define functions:

In [ ]:
def membrane_num(mem):
    return int(str(mem).replace("T", ""))


def ordered_pair_label(a, b):
    return f"T{min(membrane_num(a), membrane_num(b))}--T{max(membrane_num(a), membrane_num(b))}"


def parse_csv_path(csv_path):
    """
    Robust parser for Morphometrics distance CSVs.

    Works for names like:
      ST1_5-7_Position_17_2_trimed_components_T29.AVV_rh9.csv
      ST1_8-5_Position_5_thylakoids_morphometrics_T14.AVV_rh9.csv
      ..._T14.AVV_rh9_excluding1borders.csv

    Assumes:
      - tomogram name = parent directory name
      - source membrane = final _T<number> before .AVV_rh9...
    """
    csv_path = Path(csv_path)
    tomogram = csv_path.parent.name

    m = re.search(
        r"_(T\d+)\.AVV_rh9(?:_excluding1borders)?\.csv$",
        csv_path.name
    )
    if m is None:
        return tomogram, None

    source_membrane = m.group(1)
    return tomogram, source_membrane


def is_distance_csv(filename, use_excluding_borders=False):
    """
    Keep only Morphometrics distance CSVs.
    Exclude runtimes and non-CSV sidecar files.
    """
    name = str(filename)

    if not name.endswith(".csv"):
        return False

    if "_runtimes.csv" in name:
        return False

    if ".AVV_rh9" not in name:
        return False

    if use_excluding_borders:
        return name.endswith(".AVV_rh9_excluding1borders.csv")
    else:
        return name.endswith(".AVV_rh9.csv") and ("_excluding1borders.csv" not in name)


def find_distance_csvs(root_dir, use_excluding_borders=False):
    root_dir = Path(root_dir)
    csv_files = []

    for dp, dn, fn in os.walk(root_dir, followlinks=True):
        for f in fn:
            if is_distance_csv(f, use_excluding_borders=use_excluding_borders):
                csv_files.append(Path(dp) / f)

    return sorted(csv_files)


def find_config_near_csv(csv_path):
    """
    Search for config yaml in:
    1) same directory as csv
    2) parent directory
    """
    search_dirs = [csv_path.parent, csv_path.parent.parent]

    for directory in search_dirs:
        if not directory.exists():
            continue

        candidates = (
            sorted(directory.glob("*config*.yml")) +
            sorted(directory.glob("*config*.yaml")) +
            sorted(directory.glob("*.yml")) +
            sorted(directory.glob("*.yaml"))
        )

        for path in candidates:
            try:
                with open(path, "r") as f:
                    cfg = yaml.safe_load(f)
                if isinstance(cfg, dict) and "distance_and_orientation_measurements" in cfg:
                    return path
            except Exception:
                pass

    return None


def load_directed_pairs_from_config(config_path):
    """
    Returns directed pairs as tuples:
    {("T2", "T3"), ("T4", "T5"), ...}
    """
    if config_path is None:
        return None

    with open(config_path, "r") as f:
        cfg = yaml.safe_load(f)

    inter = cfg.get("distance_and_orientation_measurements", {}).get("inter", {})
    directed_pairs = set()

    for src, tgts in inter.items():
        if tgts is None:
            continue
        for tgt in tgts:
            directed_pairs.add((src, tgt))

    return directed_pairs


def keep_direction(source_membrane, target_membrane, directed_pairs):
    """
    If config-directed pairs exist, keep only those exact directions.
    Otherwise keep only one canonical direction:
    smaller T number -> larger T number.
    """
    if directed_pairs is not None:
        return (source_membrane, target_membrane) in directed_pairs, "config_direction"

    return membrane_num(source_membrane) < membrane_num(target_membrane), "numeric_direction"


def weighted_quantile(values, quantiles, sample_weight=None):
    values = np.asarray(values, dtype=float)
    quantiles = np.asarray(quantiles, dtype=float)

    if sample_weight is None:
        sample_weight = np.ones_like(values, dtype=float)
    else:
        sample_weight = np.asarray(sample_weight, dtype=float)

    mask = np.isfinite(values) & np.isfinite(sample_weight) & (sample_weight > 0)
    values = values[mask]
    sample_weight = sample_weight[mask]

    if values.size == 0:
        return np.full(quantiles.shape, np.nan, dtype=float)

    sorter = np.argsort(values)
    values = values[sorter]
    sample_weight = sample_weight[sorter]

    cumulative = np.cumsum(sample_weight) - 0.5 * sample_weight
    cumulative = cumulative / np.sum(sample_weight)

    return np.interp(quantiles, cumulative, values)


def summarise_distances(g):
    vals = g["distance"].to_numpy()
    areas = g["area"].to_numpy()

    q25_w, q50_w, q75_w = weighted_quantile(vals, [0.25, 0.50, 0.75], areas)

    return pd.Series({
        "n_vertices": len(g),
        "area_total": np.nansum(areas),
        "distance_mean": np.nanmean(vals),
        "distance_median": np.nanmedian(vals),
        "distance_std": np.nanstd(vals, ddof=1) if len(vals) > 1 else np.nan,
        "distance_mean_weighted": np.average(vals, weights=areas),
        "distance_median_weighted": q50_w,
        "distance_q25_weighted": q25_w,
        "distance_q75_weighted": q75_w,
    })


def collect_distance_rows(root_dir, condition, use_excluding_borders=False):
    """
    Collect all usable directed distances under one condition parent directory.

    Important:
    - membrane labels are local to each tomogram
    - each pair instance is unique within a tomogram
    - only one direction per pair is kept
    """
    root_dir = Path(root_dir)

    all_parts = []
    skipped = []
    debug_rows = []

    csv_files = find_distance_csvs(root_dir, use_excluding_borders=use_excluding_borders)

    for csv_path in csv_files:
        tomogram, source_membrane = parse_csv_path(csv_path)
        if source_membrane is None:
            skipped.append((str(csv_path), "name_parse_failed"))
            continue

        config_path = find_config_near_csv(csv_path)
        directed_pairs = load_directed_pairs_from_config(config_path)

        try:
            df = pd.read_csv(csv_path)
        except Exception as e:
            skipped.append((str(csv_path), f"read_failed: {e}"))
            continue

        dist_cols = [c for c in df.columns if c.endswith("_dist")]

        debug_entry = {
            "csv_file": str(csv_path),
            "tomogram": tomogram,
            "source_membrane": source_membrane,
            "config_file": str(config_path) if config_path else None,
            "n_dist_cols": len(dist_cols),
            "dist_cols_found": ",".join(dist_cols),
            "pair_candidates": "",
            "pairs_kept": "",
            "n_rows_in_csv": len(df),
            "n_rows_kept_total": 0,
        }

        if not dist_cols:
            skipped.append((str(csv_path), "no_distance_columns"))
            debug_rows.append(debug_entry)
            continue

        pair_candidates = []
        pairs_kept = []

        for dist_col in dist_cols:
            target_membrane = dist_col[:-5]
            pair_candidates.append(f"{source_membrane}->{target_membrane}")

            keep, rule = keep_direction(source_membrane, target_membrane, directed_pairs)
            if not keep:
                continue

            sub = df[["area", dist_col]].copy()
            sub = sub.rename(columns={dist_col: "distance_raw_nm"})
            sub = sub[np.isfinite(sub["distance_raw_nm"])].copy()

            if sub.empty:
                continue

            sub["distance"] = np.clip(sub["distance_raw_nm"] - MEMBRANE_THICKNESS_NM, 0, None)
            sub["membrane_thickness_subtracted_nm"] = MEMBRANE_THICKNESS_NM

            pair_instance_id = f"{tomogram}::{source_membrane}->{target_membrane}"
            unordered_pair_instance_id = f"{tomogram}::{ordered_pair_label(source_membrane, target_membrane)}"
            source_instance_id = f"{tomogram}::{source_membrane}"

            sub["condition"] = condition
            sub["tomogram"] = tomogram
            sub["source_membrane"] = source_membrane
            sub["target_membrane"] = target_membrane
            sub["pair_label_local"] = f"{source_membrane}->{target_membrane}"
            sub["unordered_pair_label_local"] = ordered_pair_label(source_membrane, target_membrane)
            sub["pair_instance_id"] = pair_instance_id
            sub["unordered_pair_instance_id"] = unordered_pair_instance_id
            sub["source_instance_id"] = source_instance_id
            sub["direction_rule"] = rule
            sub["csv_file"] = str(csv_path)
            sub["config_file"] = str(config_path) if config_path is not None else None

            pairs_kept.append(f"{source_membrane}->{target_membrane}")
            debug_entry["n_rows_kept_total"] += len(sub)
            all_parts.append(sub)

        debug_entry["pair_candidates"] = ",".join(pair_candidates)
        debug_entry["pairs_kept"] = ",".join(pairs_kept)
        debug_rows.append(debug_entry)

    if all_parts:
        out = pd.concat(all_parts, ignore_index=True)
    else:
        out = pd.DataFrame(columns=[
            "area", "distance_raw_nm", "distance", "membrane_thickness_subtracted_nm",
            "condition", "tomogram", "source_membrane", "target_membrane",
            "pair_label_local", "unordered_pair_label_local",
            "pair_instance_id", "unordered_pair_instance_id", "source_instance_id",
            "direction_rule", "csv_file", "config_file"
        ])

    skipped_df = pd.DataFrame(skipped, columns=["file", "reason"])
    debug_df = pd.DataFrame(debug_rows)

    return out, skipped_df, debug_df


def compare_two_conditions(df, value_col, condition_order=None):
    if condition_order is None:
        condition_order = list(CONDITION_DIRS.keys())

    a_name, b_name = condition_order
    a = df.loc[df["condition"] == a_name, value_col].dropna().to_numpy()
    b = df.loc[df["condition"] == b_name, value_col].dropna().to_numpy()

    p = np.nan
    if len(a) >= 2 and len(b) >= 2:
        _, p = mannwhitneyu(a, b, alternative="two-sided")

    return pd.DataFrame([{
        "value_col": value_col,
        f"n_{a_name}": len(a),
        f"n_{b_name}": len(b),
        f"median_{a_name}": np.median(a) if len(a) else np.nan,
        f"median_{b_name}": np.median(b) if len(b) else np.nan,
        f"mean_{a_name}": np.mean(a) if len(a) else np.nan,
        f"mean_{b_name}": np.mean(b) if len(b) else np.nan,
        f"delta_median_{a_name}_minus_{b_name}": (np.median(a) - np.median(b)) if len(a) and len(b) else np.nan,
        "p_value_mannwhitney": p,
    }])


def plot_violin_with_points(df, value_col, ylabel=None, title=None, condition_order=None):
    """
    Violin plot with jittered points for pooled summaries by condition.
    """
    if condition_order is None:
        condition_order = list(CONDITION_DIRS.keys())

    fig, ax = plt.subplots(figsize=(5.0, 4.8))
    rng = np.random.default_rng(0)

    data = []
    positions = np.arange(1, len(condition_order) + 1)

    for cond in condition_order:
        vals = df.loc[df["condition"] == cond, value_col].dropna().to_numpy()
        data.append(vals)

    violin = ax.violinplot(
        data,
        positions=positions,
        widths=0.8,
        showmeans=False,
        showmedians=True,
        showextrema=False
    )

    for body, cond in zip(violin["bodies"], condition_order):
        body.set_facecolor(CONDITION_COLORS.get(cond, "gray"))
        body.set_edgecolor(CONDITION_COLORS.get(cond, "gray"))
        body.set_alpha(0.35)

    if "cmedians" in violin:
        violin["cmedians"].set_color("black")
        violin["cmedians"].set_linewidth(1.5)

    for i, cond in enumerate(condition_order, start=1):
        vals = df.loc[df["condition"] == cond, value_col].dropna().to_numpy()
        if len(vals) == 0:
            continue

        x = rng.normal(i, 0.05, size=len(vals))
        ax.plot(
            x, vals, "o",
            ms=4.5,
            alpha=0.7,
            color=CONDITION_COLORS.get(cond, None)
        )

    ax.set_xticks(positions)
    ax.set_xticklabels(condition_order)
    ax.set_ylabel(ylabel if ylabel else value_col)
    if title:
        ax.set_title(title)
    ax.grid(axis="y", alpha=0.25)
    plt.show()


def plot_vertex_histograms(vertex_df, bins=80, density=False, xlim=None, condition_order=None):
    """
    Per-vertex pooled histogram across the full dataset.

    density=False:
        each condition histogram is normalized to its own maximum count = 1

    density=True:
        standard probability density histogram (area under curve = 1)

    xlim:
        tuple like (0, 40)
    """
    if condition_order is None:
        condition_order = list(CONDITION_DIRS.keys())

    vals_all = vertex_df["distance"].dropna().to_numpy()
    if xlim is not None:
        xmin, xmax = xlim
        vals_all = vals_all[(vals_all >= xmin) & (vals_all <= xmax)]

    if len(vals_all) == 0:
        print("No vertex distances found.")
        return

    bin_edges = np.histogram_bin_edges(vals_all, bins=bins)
    centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])

    fig, ax = plt.subplots(figsize=(7.0, 4.8))

    for cond in condition_order:
        vals = vertex_df.loc[vertex_df["condition"] == cond, "distance"].dropna().to_numpy()

        if xlim is not None:
            vals = vals[(vals >= xmin) & (vals <= xmax)]

        if len(vals) == 0:
            continue

        color = CONDITION_COLORS.get(cond, None)

        if density:
            counts, _ = np.histogram(vals, bins=bin_edges, density=True)
            y = counts
            ylabel = "density"
        else:
            counts, _ = np.histogram(vals, bins=bin_edges, density=False)
            if counts.max() == 0:
                continue
            y = counts / counts.max()
            ylabel = "normalized count (max=1)"

        ax.plot(centers, y, linewidth=2, color=color, label=f"{cond} (n={len(vals)})")
        ax.fill_between(centers, 0, y, color=color, alpha=0.2)

    ax.set_xlabel("distance (nm)")
    ax.set_ylabel(ylabel)
    ax.set_title("Per-vertex pooled distance distribution")
    if xlim is not None:
        ax.set_xlim(xlim)
    ax.legend(frameon=False)
    ax.grid(alpha=0.25)
    plt.show()

def plot_violin_with_boxplot(df, value_col, ylabel=None, title=None, condition_order=None, save_svg=None):
    """
    Violin plot with overlaid boxplot and point cloud.
    Shows both median (line in boxplot) and mean (triangle marker).
    
    Parameters:
    - df: DataFrame with 'condition' column and value_col
    - value_col: column name to plot
    - ylabel: y-axis label
    - title: plot title
    - condition_order: list of condition names in desired order
    """
    if condition_order is None:
        condition_order = list(CONDITION_DIRS.keys())

    fig, ax = plt.subplots(figsize=(5.0, 4.8))
    rng = np.random.default_rng(0)

    data = []
    positions = np.arange(1, len(condition_order) + 1)

    for cond in condition_order:
        vals = df.loc[df["condition"] == cond, value_col].dropna().to_numpy()
        data.append(vals)

    # Create violin plot
    violin = ax.violinplot(
        data,
        positions=positions,
        widths=0.7,
        showmeans=False,
        showmedians=False,
        showextrema=False
    )

    for body, cond in zip(violin["bodies"], condition_order):
        body.set_facecolor(CONDITION_COLORS.get(cond, "gray"))
        body.set_edgecolor(CONDITION_COLORS.get(cond, "gray"))
        body.set_alpha(0.25)

    # Overlay boxplot
    bp = ax.boxplot(
        data,
        positions=positions,
        widths=0.3,
        patch_artist=True,
        showfliers=False,
        boxprops=dict(linewidth=1.5),
        whiskerprops=dict(linewidth=1.5),
        capprops=dict(linewidth=1.5),
        medianprops=dict(color="black", linewidth=2)
    )

    # Color the boxes
    for patch, cond in zip(bp["boxes"], condition_order):
        patch.set_facecolor(CONDITION_COLORS.get(cond, "gray"))
        patch.set_alpha(0.7)

    # Add mean markers (triangle)
    for i, cond in enumerate(condition_order, start=1):
        vals = df.loc[df["condition"] == cond, value_col].dropna().to_numpy()
        if len(vals) > 0:
            mean_val = np.mean(vals)
            ax.plot(i, mean_val, marker="^", markersize=8, color="red", 
                   markeredgecolor="darkred", markeredgewidth=1, zorder=3, label="mean" if i == 1 else "")

    # Add individual points (jittered)
    for i, cond in enumerate(condition_order, start=1):
        vals = df.loc[df["condition"] == cond, value_col].dropna().to_numpy()
        if len(vals) == 0:
            continue

        x = rng.normal(i, 0.04, size=len(vals))
        ax.plot(
            x, vals, "o",
            ms=3.5,
            alpha=0.5,
            color=CONDITION_COLORS.get(cond, None)
        )

    ax.set_xticks(positions)
    ax.set_xticklabels(condition_order)
    ax.set_ylabel(ylabel if ylabel else value_col)
    if title:
        ax.set_title(title)
    ax.grid(axis="y", alpha=0.25)
    
    # Add legend for mean marker
    from matplotlib.lines import Line2D
    legend_elements = [Line2D([0], [0], marker="^", color="w", markerfacecolor="red", 
                             markeredgecolor="darkred", markersize=8, label="Mean")]
    ax.legend(handles=legend_elements, loc="best", frameon=False)
    
    if save_svg:
       plt.savefig(save_svg, format='svg', dpi=300, bbox_inches='tight')
       print(f"✓ Saved to {save_svg}")
   

       
    plt.show()


#### Gather data and debug:

In [ ]:
for condition, root in CONDITION_DIRS.items():
    print(f"\n{condition}: {root}")
    print("exists:", root.exists())
    print("is_dir:", root.is_dir())

    if root.exists():
        files = find_distance_csvs(
            root,
            use_excluding_borders=USE_EXCLUDING_BORDERS
        )
        print("matching distance csv files:", len(files))

In [ ]:
all_vertex_dfs = []
all_skipped_dfs = []
all_debug_dfs = []

for condition, root_dir in CONDITION_DIRS.items():
    vertex_df_cond, skipped_df_cond, debug_df_cond = collect_distance_rows(
        root_dir=root_dir,
        condition=condition,
        use_excluding_borders=USE_EXCLUDING_BORDERS
    )

    print(f"\nCondition: {condition}")
    print(f"  root: {root_dir}")
    print(f"  csv files checked: {len(debug_df_cond)}")
    print(f"  usable rows: {len(vertex_df_cond)}")

    all_vertex_dfs.append(vertex_df_cond)
    all_skipped_dfs.append(skipped_df_cond)
    all_debug_dfs.append(debug_df_cond)

vertex_df = pd.concat(all_vertex_dfs, ignore_index=True)
skipped_df = pd.concat(all_skipped_dfs, ignore_index=True)
debug_df = pd.concat(all_debug_dfs, ignore_index=True)

# Apply distance cutoff filter if specified
if DISTANCE_CUTOFF_NM is not None:
    n_before = len(vertex_df)
    vertex_df = vertex_df[vertex_df["distance"] <= DISTANCE_CUTOFF_NM].copy()
    n_after = len(vertex_df)
    n_removed = n_before - n_after
    print(f"\n⚠ Distance cutoff applied: DISTANCE_CUTOFF_NM = {DISTANCE_CUTOFF_NM} nm")
    print(f"   Removed {n_removed} vertices ({100*n_removed/n_before:.1f}%) beyond cutoff")
    print(f"   Retained: {n_after} vertices")

print("\nTotal usable rows:", len(vertex_df))
print("Unique tomograms:", vertex_df["tomogram"].nunique())
print("Unique pair instances:", vertex_df["pair_instance_id"].nunique())

vertex_df.to_csv(OUTDIR / "all_directed_vertex_distances.csv", index=False)
debug_df.to_csv(OUTDIR / "distance_debug_table.csv", index=False)
skipped_df.to_csv(OUTDIR / "distance_skipped_files.csv", index=False)

In [ ]:
pair_instance_summary_df = (
    vertex_df
    .groupby(
        ["condition", "tomogram", "source_membrane", "target_membrane",
         "pair_label_local", "unordered_pair_label_local",
         "pair_instance_id", "unordered_pair_instance_id"],
        group_keys=False
    )
    .apply(summarise_distances)
    .reset_index()
)

display(pair_instance_summary_df.head())

pair_instance_summary_df.to_csv(
    OUTDIR / "distance_summary_per_pair_instance.csv",
    index=False
)

In [ ]:
tomogram_summary_df = (
    vertex_df
    .groupby(["condition", "tomogram"], group_keys=False)
    .apply(summarise_distances)
    .reset_index()
)

display(tomogram_summary_df.head())

tomogram_summary_df.to_csv(
    OUTDIR / "distance_summary_per_tomogram.csv",
    index=False
)

In [ ]:
source_membrane_summary_df = (
    vertex_df
    .groupby(
        ["condition", "tomogram", "source_membrane", "source_instance_id"],
        group_keys=False
    )
    .apply(summarise_distances)
    .reset_index()
)

display(source_membrane_summary_df.head())

source_membrane_summary_df.to_csv(
    OUTDIR / "distance_summary_per_source_membrane.csv",
    index=False
)

#### **Plots:**

In [ ]:
plot_vertex_histograms(
    vertex_df,
    bins=100,
    density=True,
    xlim=(0, 125),
    
)

In [ ]:
plot_violin_with_points(
    source_membrane_summary_df,
    value_col="distance_median",
    ylabel="Median distance per membrane (nm)",
    title="Interthylakoid distance: per membrane"
)

source_membrane_stats_df = compare_two_conditions(
    source_membrane_summary_df,
    value_col="distance_median"
)

display(source_membrane_stats_df)
source_membrane_stats_df.to_csv(
    OUTDIR / "stats_source_median",
    index=False
)

In [ ]:
plot_violin_with_boxplot(
    source_membrane_summary_df,
    value_col="distance_median",
    ylabel="Median distance per membrane (nm)",
    title="Stromal gap size: per membrane (violin + boxplot + mean)"
)

# Also show mean-based analysis for comparison
plot_violin_with_boxplot(
    source_membrane_summary_df,
    value_col="distance_mean",
    ylabel="Mean distance per membrane (nm)",
    title="Interthylakoid distance (nm)",
    save_svg="mean_interthylakoid_distance_FRLWL.svg"
)
